# 01 — Train ESN upsampler (2 Hz → 100 Hz+)

Uses existing modules:
- `s2r.nodes.esn_engine.EchoStateNetwork`
- `s2r.experiments.esn_data`
- `s2r.training.train_esn`

Data sources: `data/raw/*.jsonl` and/or `data/benchmark/tasks/*/episodes`.

In [ ]:
from pathlib import Path
import sys
sys.path.insert(0, str(Path.cwd() / "_lib"))
from bootstrap import setup
ROOT = setup()

import numpy as np
import matplotlib.pyplot as plt

from s2r.core.config import load_config
from s2r.experiments.paths import ESN_DATA, RAW, MODELS, ensure_experiment_dirs
from s2r.experiments.esn_data import export_esn_split, load_pairs
from s2r.nodes.esn_engine import EchoStateNetwork

ensure_experiment_dirs()
cfg = load_config(ROOT / "config" / "default.yaml")
n_joints = int(cfg["robot"]["n_joints"])
esn_cfg = cfg["esn"]
print("n_joints", n_joints, "raw exists", RAW.exists())

In [ ]:
# Build train/val from collected episodes (pipeline JSONL)
paths = export_esn_split(n_joints=n_joints, sources=[RAW], val_ratio=0.2)
paths

In [ ]:
Xtr, Ytr = load_pairs("train")
Xva, Yva = load_pairs("val")
print(Xtr.shape, Ytr.shape, Xva.shape, Yva.shape)

esn = EchoStateNetwork(
    n_inputs=n_joints,
    n_outputs=n_joints,
    reservoir_size=int(esn_cfg.get("reservoir_size", 300)),
    spectral_radius=float(esn_cfg.get("spectral_radius", 0.9)),
    sparsity=float(esn_cfg.get("sparsity", 0.1)),
    input_scale=float(esn_cfg.get("input_scale", 0.5)),
    leaking_rate=float(esn_cfg.get("leaking_rate", 0.3)),
)
mse = esn.fit_ridge(Xtr, Ytr, washout=int(esn_cfg.get("washout", 20)))
print("train MSE", mse)

In [ ]:
# Validation rollout error
esn.reset()
preds = np.array([esn.update(u) for u in Xva])
val_mse = float(np.mean((preds - Yva) ** 2))
print("val MSE", val_mse)

fig, ax = plt.subplots(figsize=(10, 3))
j = 0
ax.plot(Yva[:200, j], label="target q0")
ax.plot(preds[:200, j], label="esn q0", alpha=0.8)
ax.legend(); ax.set_title("ESN joint-0 validation preview")
plt.show()

curve_path = ESN_DATA / "curves" / "last_train.json"
import json
curve_path.write_text(json.dumps({"train_mse": mse, "val_mse": val_mse}, indent=2))
print("wrote", curve_path)

In [ ]:
# Save checkpoint for pipeline deploy
out = ESN_DATA / "checkpoints" / "esn_upsample.npz"
deploy = MODELS / "esn_upsample.npz"
esn.save(out)
esn.save(deploy)
print("saved", out)
print("deploy copy", deploy)